In [1]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub librosa

# =========================================================
# IMPORTS
# =========================================================
import os
import random
import numpy as np
import torch
import pandas as pd
import evaluate
import librosa

from huggingface_hub import login

from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

# =========================================================
# HF LOGIN
# Prompts you to paste your token at runtime (hidden input, like a
# PIN) instead of hardcoding it in the script.
# =========================================================
from getpass import getpass

login(getpass("Enter your Hugging Face token: "))

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"
test_csv = "SPCS/manifests/test.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET CONVERSION
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# Switch model_name to "openai/whisper-tiny" if small keeps
# overfitting hard even after the regularization changes below.
# =========================================================
model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

# ---- Regularization additions ----
# SpecAugment: randomly masks time/frequency bands in the input
# spectrogram during training, forcing the model to not rely on
# any single narrow slice of audio. Cheap and effective against
# overfitting on small speech datasets.
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_time_length = 10
model.config.mask_feature_prob = 0.05
model.config.mask_feature_length = 10

# Optional: freeze the encoder if your training set is very small
# (e.g. a few hours of audio). This forces the model to only adapt
# the decoder/language side, which is much less prone to overfitting
# on limited data. Uncomment if train loss still crashes to ~0 fast.
# model.freeze_encoder()

model.to(device)

# =========================================================
# SPEED AUGMENTATION (new)
# Randomly speeds up or slows down training audio before feature
# extraction. This is a standard, cheap augmentation for ASR
# (commonly called "speed perturbation") that exposes the model to
# faster/slower speaking rates, helping it generalize better and
# reducing overfitting on a small dataset. Applied to TRAINING data
# only - validation audio must stay unmodified so your WER/CER stay
# a fair, untouched measure of real performance.
# =========================================================

SPEED_FACTORS = [0.9, 1.0, 1.1]  

def apply_speed_augmentation(audio_array, sampling_rate):
    speed_factor = random.choice(SPEED_FACTORS)
    if speed_factor == 1.0:
        return audio_array
    
    return librosa.effects.time_stretch(audio_array.astype(np.float32), rate=speed_factor)

# =========================================================
# PREPROCESSING
# =========================================================
def prepare_dataset_train(batch):
    audio = batch["audio"]

    audio_array = apply_speed_augmentation(audio["array"], audio["sampling_rate"])

    batch["input_features"] = processor.feature_extractor(
        audio_array,
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

def prepare_dataset_eval(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

train_ds = train_ds.map(
    prepare_dataset_train,
    remove_columns=train_ds.column_names,
    num_proc=1
)

valid_ds = valid_ds.map(
    prepare_dataset_eval,
    remove_columns=valid_ds.column_names,
    num_proc=1
)

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# WER + CER METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

# =========================================================
# TRAINING CONFIG (FIXED - anti-overfitting)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models_result/whisper_speed_aug",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch size 8

    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,               # smooth ramp-up instead of jumping straight to peak LR
    weight_decay=0.01,              # penalizes large weights, directly fights overfitting

    num_train_epochs=10,            # kept high on purpose - early stopping will cut it short
    max_grad_norm=1.0,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=3,

    predict_with_generate=True,
    generation_max_length=225,

    # Keep the checkpoint with the best validation WER, not the last one.
    # This alone fixes most of the damage from overfitting: even if
    # training runs long, you deploy the epoch that generalized best.
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # NOTE: safe_serialization is handled manually at save time below
    # (this transformers version doesn't accept save_safetensors here).

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
    # Stops training if validation WER hasn't improved for 3 straight
    # evals - prevents the "train loss -> 0, val loss flat" pattern
    # you saw from running all the way to epoch 10.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL (this saves the BEST checkpoint, thanks to
# load_best_model_at_end=True above)
# =========================================================
save_path = "SPCS/models_result/whisper_speed_aug"

model.tie_weights()  # make sure proj_out is explicitly linked to the embedding weights

# Saving directly via model.save_pretrained (rather than trainer.save_model)
# so we can pass safe_serialization=False - this avoids the "missing keys:
# proj_out.weight" warning caused by Whisper's tied embedding/output layer
# being dropped from safetensors checkpoints on save.
model.save_pretrained(save_path, safe_serialization=False)
processor.save_pretrained(save_path)

# =========================================================
# SAVE TRAINER STATE (training loss / val loss / wer / cer history)
# Writes trainer_state.json into save_path so the full log_history
# is reloadable later, instead of only living in checkpoint-N folders.
# =========================================================
trainer.save_state()

log_history = trainer.state.log_history
log_df = pd.DataFrame(log_history)
log_csv_path = os.path.join(save_path, "training_log_history.csv")
log_df.to_csv(log_csv_path, index=False)

print("\nTraining complete.")
print("Best model (by validation WER) saved to:", save_path)
print("Trainer state saved to:", save_path)
print("Training log history exported to:", log_csv_path)

# =========================================================
# TEST SET EVALUATION (NEW)
# Runs the best saved checkpoint on the held-out test set - data
# never seen during training or used for checkpoint selection.
# This gives the final, unbiased WER/CER for the speed-augmented model.
# =========================================================
test_df = pd.read_csv(test_csv)
print("\nTest samples:", len(test_df))

test_ds = Dataset.from_pandas(test_df)
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))
test_ds = test_ds.map(prepare_dataset_eval, remove_columns=test_ds.column_names, num_proc=1)

test_results = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test")

print("\nTest set results:")
print(test_results)

# ---- Save test results to CSV ----
test_results_df = pd.DataFrame([test_results])
test_results_csv_path = os.path.join(save_path, "test_results.csv")
test_results_df.to_csv(test_results_csv_path, index=False)
print("Test results saved to:", test_results_csv_path)

Enter your Hugging Face token:  ········


Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

...


Training complete.
Best model (by validation WER) saved to: SPCS/models_result/whisper_speed_aug
Trainer state saved to: SPCS/models_result/whisper_speed_aug
Training log history exported to: SPCS/models_result/whisper_speed_aug/training_log_history.csv

Test samples: 1239


Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

[transformers] early stopping required metric_for_best_model, but did not find eval_wer so early stopping is disabled


Training Loss,Validation Loss,Epoch,Wer,Cer
0.013072,0.010337,10,0.035398,0.026848



Test set results:
{'test_loss': 0.010336576029658318, 'test_wer': 0.035398230088495575, 'test_cer': 0.02684779120761579}
Test results saved to: SPCS/models_result/whisper_speed_aug/test_results.csv


In [10]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ---- Results data ----
data = {
    "Epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "Training Loss": [2.920721, 0.831730, 0.314292, 0.127693, 0.105760,
                       0.038458, 0.042406, 0.036769, 0.024118, 0.033198],
    "Validation Loss": [0.257368, 0.070914, 0.038083, 0.024361, 0.018908,
                         0.014929, 0.012923, 0.011995, 0.011878, 0.011377],
    "Wer": [0.316176, 0.097243, 0.056985, 0.034191, 0.025735,
            0.060110, 0.020956, 0.018750, 0.030699, 0.018199],
    "Cer": [0.148335, 0.048454, 0.032466, 0.016688, 0.012804,
            0.032326, 0.016548, 0.009551, 0.016513, 0.010076],
}

df = pd.DataFrame(data)

# Round all numeric columns to 3 decimal places
df = df.round(4)

# ---- Output directory ----
output_dir = "SPCS/graph_results/noise_whisper"
os.makedirs(output_dir, exist_ok=True)

# Save table as CSV
csv_path = os.path.join(output_dir, "noise_whisper_results.csv")
df.to_csv(csv_path, index=False)
print(f"Saved table to: {csv_path}")

# ---- Save a table image ----
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis("off")
table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    cellLoc="center",
    loc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
plt.title("Noise Whisper - Training Results", pad=15)
plt.tight_layout()

table_img_path = os.path.join(output_dir, "noise_whisper_table.png")
plt.savefig(table_img_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved table image to: {table_img_path}")

# ---- Save a metrics plot (loss/WER/CER over epochs) ----
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(df["Epoch"], df["Training Loss"], marker="o", label="Training Loss")
ax1.plot(df["Epoch"], df["Validation Loss"], marker="o", label="Validation Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(df["Epoch"], df["Wer"], marker="s", linestyle="--", color="green", label="WER")
ax2.plot(df["Epoch"], df["Cer"], marker="s", linestyle="--", color="red", label="CER")
ax2.set_ylabel("WER / CER")
ax2.legend(loc="upper right")

plt.title("Noise Whisper - Loss & Error Rates")
plt.tight_layout()

plot_path = os.path.join(output_dir, "noise_whisper_plot.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved plot to: {plot_path}")

print("\nFinal rounded table:")
print(df.to_string(index=False))

Saved table to: SPCS/graph_results/noise_whisper/noise_whisper_results.csv
Saved table image to: SPCS/graph_results/noise_whisper/noise_whisper_table.png
Saved plot to: SPCS/graph_results/noise_whisper/noise_whisper_plot.png

Final rounded table:
 Epoch  Training Loss  Validation Loss    Wer    Cer
     1         2.9207           0.2574 0.3162 0.1483
     2         0.8317           0.0709 0.0972 0.0485
     3         0.3143           0.0381 0.0570 0.0325
     4         0.1277           0.0244 0.0342 0.0167
     5         0.1058           0.0189 0.0257 0.0128
     6         0.0385           0.0149 0.0601 0.0323
     7         0.0424           0.0129 0.0210 0.0165
     8         0.0368           0.0120 0.0188 0.0096
     9         0.0241           0.0119 0.0307 0.0165
    10         0.0332           0.0114 0.0182 0.0101


In [6]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ---- Results data ----
data = {
    "Epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "Training Loss": [5.029433, 1.392660, 0.537004, 0.184045, 0.160115,
                       0.073033, 0.070284, 0.015459, 0.059944, 0.039632],
    "Validation Loss": [0.288331, 0.072325, 0.034518, 0.022580, 0.015390,
                         0.012886, 0.010810, 0.008835, 0.008969, 0.009214],
    "Wer": [0.385846, 0.191912, 0.116728, 0.046140, 0.031434,
            0.027022, 0.027390, 0.027206, 0.023364, 0.028125],
    "Cer": [0.205779, 0.155052, 0.067800, 0.031556, 0.016128,
            0.013644, 0.015148, 0.014694, 0.012385, 0.014973],
}

df = pd.DataFrame(data)

# Round all numeric columns to 4 decimal places
df = df.round(4)

# ---- Output directory ----
output_dir = "SPCS/graph_results/pitch_whisper"
os.makedirs(output_dir, exist_ok=True)

# Save table as CSV
csv_path = os.path.join(output_dir, "pitch_whisper_results.csv")
df.to_csv(csv_path, index=False)
print(f"Saved table to: {csv_path}")

# ---- Save a table image ----
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis("off")
table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    cellLoc="center",
    loc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
plt.title("Whisper Pitch Augmentation - Training Results", pad=15)
plt.tight_layout()

table_img_path = os.path.join(output_dir, "pitch_whisper_table.png")
plt.savefig(table_img_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved table image to: {table_img_path}")

# ---- Save a metrics plot (loss/WER/CER over epochs) ----
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(df["Epoch"], df["Training Loss"], marker="o", label="Training Loss")
ax1.plot(df["Epoch"], df["Validation Loss"], marker="o", label="Validation Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(df["Epoch"], df["Wer"], marker="s", linestyle="--", color="green", label="WER")
ax2.plot(df["Epoch"], df["Cer"], marker="s", linestyle="--", color="red", label="CER")
ax2.set_ylabel("WER / CER")
ax2.legend(loc="upper right")

plt.title("Whisper Pitch Augmentation - Loss & Error Rates")
plt.tight_layout()

plot_path = os.path.join(output_dir, "pitch_whisper_plot.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved plot to: {plot_path}")

print("\nFinal rounded table:")
print(df.to_string(index=False))

Saved table to: SPCS/graph_results/pitch_whisper/pitch_whisper_results.csv
Saved table image to: SPCS/graph_results/pitch_whisper/pitch_whisper_table.png
Saved plot to: SPCS/graph_results/pitch_whisper/pitch_whisper_plot.png

Final rounded table:
 Epoch  Training Loss  Validation Loss    Wer    Cer
     1         5.0294           0.2883 0.3858 0.2058
     2         1.3927           0.0723 0.1919 0.1551
     3         0.5370           0.0345 0.1167 0.0678
     4         0.1840           0.0226 0.0461 0.0316
     5         0.1601           0.0154 0.0314 0.0161
     6         0.0730           0.0129 0.0270 0.0136
     7         0.0703           0.0108 0.0274 0.0151
     8         0.0155           0.0088 0.0272 0.0147
     9         0.0599           0.0090 0.0234 0.0124
    10         0.0396           0.0092 0.0281 0.0150


In [9]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ---- Results data ----
data = {
    "Epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "Training Loss": [3.207616, 0.802069, 0.375485, 0.201009, 0.105445,
                       0.193568, 0.064287, 0.062874, 0.057113, 0.014460],
    "Validation Loss": [0.258704, 0.066238, 0.033915, 0.020546, 0.018933,
                         0.013757, 0.011377, 0.010734, 0.009982, 0.010033],
    "Wer": [0.332537, 0.129412, 0.047978, 0.029963, 0.025919,
            0.021875, 0.016728, 0.017279, 0.016912, 0.018015],
    "Cer": [0.181850, 0.063567, 0.023650, 0.014833, 0.019696,
            0.018962, 0.015218, 0.022950, 0.016093, 0.008991],
}

df = pd.DataFrame(data)

# Round all numeric columns to 3 decimal places
df = df.round(4)

# ---- Output directory ----
output_dir = "SPCS/graph_results/specaugment_whisper"
os.makedirs(output_dir, exist_ok=True)

# Save table as CSV
csv_path = os.path.join(output_dir, "specaugment_whisper_results.csv")
df.to_csv(csv_path, index=False)
print(f"Saved table to: {csv_path}")

# ---- Save a table image ----
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis("off")
table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    cellLoc="center",
    loc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
plt.title("SpecAugment Whisper - Training Results", pad=15)
plt.tight_layout()

table_img_path = os.path.join(output_dir, "specaugment_whisper_table.png")
plt.savefig(table_img_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved table image to: {table_img_path}")

# ---- Save a metrics plot (loss/WER/CER over epochs) ----
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(df["Epoch"], df["Training Loss"], marker="o", label="Training Loss")
ax1.plot(df["Epoch"], df["Validation Loss"], marker="o", label="Validation Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(df["Epoch"], df["Wer"], marker="s", linestyle="--", color="green", label="WER")
ax2.plot(df["Epoch"], df["Cer"], marker="s", linestyle="--", color="red", label="CER")
ax2.set_ylabel("WER / CER")
ax2.legend(loc="upper right")

plt.title("SpecAugment Whisper - Loss & Error Rates")
plt.tight_layout()

plot_path = os.path.join(output_dir, "specaugment_whisper_plot.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved plot to: {plot_path}")

print("\nFinal rounded table:")
print(df.to_string(index=False))

Saved table to: SPCS/graph_results/specaugment_whisper/specaugment_whisper_results.csv
Saved table image to: SPCS/graph_results/specaugment_whisper/specaugment_whisper_table.png
Saved plot to: SPCS/graph_results/specaugment_whisper/specaugment_whisper_plot.png

Final rounded table:
 Epoch  Training Loss  Validation Loss    Wer    Cer
     1         3.2076           0.2587 0.3325 0.1819
     2         0.8021           0.0662 0.1294 0.0636
     3         0.3755           0.0339 0.0480 0.0236
     4         0.2010           0.0205 0.0300 0.0148
     5         0.1054           0.0189 0.0259 0.0197
     6         0.1936           0.0138 0.0219 0.0190
     7         0.0643           0.0114 0.0167 0.0152
     8         0.0629           0.0107 0.0173 0.0230
     9         0.0571           0.0100 0.0169 0.0161
    10         0.0145           0.0100 0.0180 0.0090


In [8]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ---- Results data ----
data = {
    "Epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "Training Loss": [2.920721, 0.831730, 0.314292, 0.127693, 0.105760,
                       0.038458, 0.042406, 0.036769, 0.024118, 0.033198],
    "Validation Loss": [0.257368, 0.070914, 0.038083, 0.024361, 0.018908,
                         0.014929, 0.012923, 0.011995, 0.011878, 0.011377],
    "Wer": [0.316176, 0.097243, 0.056985, 0.034191, 0.025735,
            0.060110, 0.020956, 0.018750, 0.030699, 0.018199],
    "Cer": [0.148335, 0.048454, 0.032466, 0.016688, 0.012804,
            0.032326, 0.016548, 0.009551, 0.016513, 0.010076],
}

df = pd.DataFrame(data)

# Round all numeric columns to 3 decimal places
df = df.round(4)

# ---- Output directory ----
output_dir = "SPCS/graph_results/noise_whisper"
os.makedirs(output_dir, exist_ok=True)

# Save table as CSV
csv_path = os.path.join(output_dir, "noise_whisper_results.csv")
df.to_csv(csv_path, index=False)
print(f"Saved table to: {csv_path}")

# ---- Save a table image ----
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis("off")
table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    cellLoc="center",
    loc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
plt.title("Noise Whisper - Training Results", pad=15)
plt.tight_layout()

table_img_path = os.path.join(output_dir, "noise_whisper_table.png")
plt.savefig(table_img_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved table image to: {table_img_path}")

# ---- Save a metrics plot (loss/WER/CER over epochs) ----
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(df["Epoch"], df["Training Loss"], marker="o", label="Training Loss")
ax1.plot(df["Epoch"], df["Validation Loss"], marker="o", label="Validation Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(df["Epoch"], df["Wer"], marker="s", linestyle="--", color="green", label="WER")
ax2.plot(df["Epoch"], df["Cer"], marker="s", linestyle="--", color="red", label="CER")
ax2.set_ylabel("WER / CER")
ax2.legend(loc="upper right")

plt.title("Noise Whisper - Loss & Error Rates")
plt.tight_layout()

plot_path = os.path.join(output_dir, "noise_whisper_plot.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved plot to: {plot_path}")

print("\nFinal rounded table:")
print(df.to_string(index=False))

Saved table to: SPCS/graph_results/noise_whisper/noise_whisper_results.csv
Saved table image to: SPCS/graph_results/noise_whisper/noise_whisper_table.png
Saved plot to: SPCS/graph_results/noise_whisper/noise_whisper_plot.png

Final rounded table:
 Epoch  Training Loss  Validation Loss    Wer    Cer
     1         2.9207           0.2574 0.3162 0.1483
     2         0.8317           0.0709 0.0972 0.0485
     3         0.3143           0.0381 0.0570 0.0325
     4         0.1277           0.0244 0.0342 0.0167
     5         0.1058           0.0189 0.0257 0.0128
     6         0.0385           0.0149 0.0601 0.0323
     7         0.0424           0.0129 0.0210 0.0165
     8         0.0368           0.0120 0.0188 0.0096
     9         0.0241           0.0119 0.0307 0.0165
    10         0.0332           0.0114 0.0182 0.0101
